# Using the Resume Feature and Iteration Loggers in QMCPy

Original QMCPy demo: [`QMCPy/demos/demo_resume_data/resume_examples.ipynb`](../../../QMCPy/demos/demo_resume_data/resume_examples.ipynb)

This Julia notebook keeps the same compact, recipe-style structure while using the `QMC.jl` APIs for resumable integration and iteration logging.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QMCSoftware/QMC.jl/blob/develop/demos/demo_resume_data/resume_examples.ipynb)

Parity note: the QMCPy notebook resumes from `1e-4` down to `1e-7`. For this Julia example `1e-7` exceeds the current lattice sample window, so the checked-in notebook uses `1e-4` down to `2.5e-5` while preserving the same resume workflow and then adds one explicit iteration-log inspection step.


In [1]:
using QMC
using Serialization
using Printf


## Step 1: Quick Estimate (Loose Tolerance)

Define a 3-D Genz integral over the unit cube and run `CubQMCLatticeG` with a loose tolerance to get a fast initial estimate. We fix the seed so notebook output is reproducible.


In [2]:
function make_solver(; abs_tol=1e-4, rel_tol=0.0, seed=7, dimension=3)
    dd = Lattice(dimension; seed=seed)
    tm = Uniform(dd)
    f = Genz(tm; kind=:oscillatory, a=ones(dimension), u=0.5 .* ones(dimension))
    return CubQMCLatticeG(f; abs_tol=abs_tol, rel_tol=rel_tol, trace_iterations=true)
end

solver_loose = make_solver(abs_tol=1e-4)
result1 = integrate(solver_loose)
@printf("Loose run: solution = %.8f, n_total = %d, error_bound = %.3e
", result1.solution, result1.data[:n_total], result1.data[:error_bound])
iterations(result1.data[:iteration_log])


Loose run: solution = -0.06235934, n_total = 65536, error_bound = 5.535e-05


iter,n,solution,err_bound,tol,time(s)
1,1024,-0.062397715,2.656e-03,1.000e-04,0.019
2,2048,-0.062498224,1.383e-03,1.000e-04,0.019
3,4096,-0.062423156,6.555e-04,1.000e-04,0.019
4,8192,-0.062328353,3.818e-04,1.000e-04,0.020
5,16384,-0.062307875,2.032e-04,1.000e-04,0.020
6,32768,-0.062335033,1.060e-04,1.000e-04,0.022
7,65536,-0.062359339,5.535e-05,1.000e-04,0.025


## Step 2: Save the Integration State

In `QMC.jl`, the resume payload is the solver data dictionary returned by `integrate`. We save it to disk with Julia serialization so a later run can pick up from the same state.


In [3]:
output_dir = joinpath(pwd(), "output")
mkpath(output_dir)
save_path = joinpath(output_dir, "resume_example_data1.jls")
serialize(save_path, result1.data)
println("Saved resume data to: ", save_path)


Saved resume data to: /Users/terrya/Documents/ProgramData/QMCSoftware_space/QMC.jl/demos/demo_resume_data/output/resume_example_data1.jls

## Step 3: Resume with a Tighter Tolerance

Load the saved state and resume with a compatible solver. Only the extra work needed for the tighter tolerance is generated.


In [4]:
loaded_data = deserialize(save_path)
solver_tight = make_solver(abs_tol=2.5e-5)
result2 = integrate(solver_tight; resume=loaded_data)
extra_samples = result2.data[:n_total] - result1.data[:n_total]
@printf("Resumed run: solution = %.8f, n_total = %d, added samples = %d
", result2.solution, result2.data[:n_total], extra_samples)
iterations(result2.data[:iteration_log])


Resumed run: solution = -0.06236067, n_total = 262144, added samples = 196608


iter,n,solution,err_bound,tol,time(s)
1,131072,-0.062361949,2.996e-05,2.500e-05,0.039
2,262144,-0.062360665,1.578e-05,2.500e-05,0.073


## Step 4: Compare with a Fresh Run

For reference, solve the same tighter problem from scratch and compare the result with the resumed workflow.


In [5]:
fresh_tight = integrate(make_solver(abs_tol=2.5e-5))
@printf("Fresh tight run: solution = %.8f, n_total = %d
", fresh_tight.solution, fresh_tight.data[:n_total])
@printf("Absolute difference between resumed and fresh solutions: %.3e
", abs(result2.solution - fresh_tight.solution))

@assert abs(result2.solution - fresh_tight.solution) ≤ result2.data[:error_bound] + fresh_tight.data[:error_bound]
@assert result2.data[:n_total] == fresh_tight.data[:n_total]
@assert !isempty(iterations(result2.data[:iteration_log]))


Fresh tight run: solution = -0.06236067, n_total = 262144
Absolute difference between resumed and fresh solutions: 0.000e+00


**Julia extension: Review the Stored Iteration Log.**

The iteration log is available even if you do not print it live. This makes it easy to inspect the stopping history after the solve completes.


In [6]:
rows = iterations(result2.data[:iteration_log])
@assert all(row -> row.n > 0, rows)
rows


iter,n,solution,err_bound,tol,time(s)
1,131072,-0.062361949,2.996e-05,2.500e-05,0.039
2,262144,-0.062360665,1.578e-05,2.500e-05,0.073


## Key Takeaways

Resume avoids repeating previously generated samples, and the stored iteration log makes the solver history reviewable after the fact.
